In [ ]:
import os
import json
import numpy as np
import pandas as pd
import sympy as sp

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
MODELSDIR = CONFIGS['filepaths']['models']
SRMODELS  = CONFIGS['experiments']['sr']['optimizedeqs']

with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)

regdf = pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv'))
REGISTRY = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']),
                              train_loss=row['train_loss'],valid_loss=row['valid_loss'])
             for _,row in regdf.iterrows()}

ORDER  = [name for name in SRMODELS if name in REGISTRY]
LABELS = {name:SRMODELS[name]['description'] for name in ORDER}

TPMEAN = STATS['tp_mean']
TPSTD  = STATS['tp_std']
ZMIN   = (0.0 - TPMEAN) / TPSTD

print(f'tp: mean={TPMEAN:.6f}, std={TPSTD:.6f}, zmin={ZMIN:.4f}')
for var in ['rh','thetae','thetaestar','bl','shf','lhf']:
    m = STATS.get(f'{var}_mean')
    s = STATS.get(f'{var}_std')
    if m is not None:
        print(f'{var}: mean={m:.6f}, std={s:.6f}')

In [ ]:
c = REGISTRY['sr_bl_eq']['constants']
blmean = STATS['bl_mean']
blstd  = STATS['bl_std']

threshold = blmean - c['a'] * blstd

print(f'SR-BL: raw = (bl_norm + {c["a"]:.4f})^3 + {c["b"]:.4f}')
print(f'Physical: raw = ((B_L - {threshold:.4f}) / {blstd:.6f})^3 + {c["b"]:.4f}')
print(f'Onset threshold: B_L = {threshold:.4f}')

In [ ]:
c = REGISTRY['sr_atm_eq']['constants']
rhmean  = STATS['rh_mean']
rhstd   = STATS['rh_std']
temean  = STATS['thetae_mean']
testd   = STATS['thetae_std']
tesmean = STATS['thetaestar_mean']
tesstd  = STATS['thetaestar_std']

tecoef  = c['b'] / testd
tescoef = c['b'] * 1.0 / tesstd
offset  = c['c'] + c['b'] * temean / testd - c['b'] * tesmean / tesstd

print(f'SR-ATM: raw = {REGISTRY["sr_atm_eq"]["form"]}')
print(f'Constants: {c}')
print()
print(f'Moisture pathway:    (RH - {rhmean:.4f}) / {rhstd:.4f}')
print(f'Instability pathway: {tecoef:.6f}*thetae - {tescoef:.6f}*thetaestar - {offset:.4f}')
print(f'thetae/thetaestar sensitivity ratio: {tecoef/tescoef:.4f}')

In [ ]:
print(f'Prediction pipeline:')
print(f'  z = {ZMIN:.4f} + max(raw, 0)')
print(f'  P = exp(z * {TPSTD:.6f} + {TPMEAN:.6f}) - 1  [mm]')
print()
for name in ORDER:
    entry = REGISTRY[name]
    print(f'{LABELS[name]}: {entry["form"]}  {entry["constants"]}')

In [ ]:
c = REGISTRY['sr_atm_eq']['constants']
rhmean  = STATS['rh_mean']
rhstd   = STATS['rh_std']
temean  = STATS['thetae_mean']
testd   = STATS['thetae_std']
tesmean = STATS['thetaestar_mean']
tesstd  = STATS['thetaestar_std']

arh  = 1.0 / rhstd
ate  = c['b'] / testd
ates = c['b'] / tesstd
aconst = c['c'] + c['b'] * temean / testd - c['b'] * tesmean / tesstd - rhmean / rhstd

print(f'Regime boundary in normalized space:')
print(f'  rh_norm = thetae_norm - {c["b"]:.4f}*thetaestar_norm - {c["c"]:.4f}')
print()
print(f'Regime boundary in physical space:')
print(f'  {arh:.6f}*RH = {ate:.6f}*thetae - {ates:.6f}*thetaestar - {aconst:.4f}')